<a href="https://colab.research.google.com/github/gabrielcabanilla1/EXSU-500-Clinical-NLP-large-language-models-rct/blob/main/notebooks/02b_prepare_200k_filtered.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02b – Extra training data from PubMed 200k RCT (filtered)
Builds a larger training set from PubMed 200k RCT for an additional experiment.
Almost all validation and test abstracts of PubMed 20k RCT also appear in the 200k training set,
so every abstract of our validation and test sets is removed before training (no leakage).
Our validation and test sets stay exactly the same (from the 20k version).

In [2]:
# ===== CONFIGURATION =====
SOURCE_VERSION = 'PubMed_200k_RCT'
EVAL_VERSION = 'PubMed_20k_RCT'

PROJECT_DIR = '/content/drive/MyDrive/exsu_project'
PREPARED_DIR = f'{PROJECT_DIR}/prepared_data'
OUTPUT_PATH = f'{PREPARED_DIR}/{SOURCE_VERSION}_train_filtered.csv'
CLASSES = ['BACKGROUND', 'OBJECTIVE', 'METHODS', 'RESULTS', 'CONCLUSIONS']
N_POSITION_BINS = 10

In [3]:
import os
from google.colab import drive

drive.mount('/content/drive')

if not os.path.exists('pubmed-rct'):
    !git clone --depth 1 https://github.com/Franck-Dernoncourt/pubmed-rct.git

SOURCE_DIR = f'pubmed-rct/{SOURCE_VERSION}'
if not os.path.exists(f'{SOURCE_DIR}/train.txt'):
    !apt-get -qq install -y p7zip-full > /dev/null
    !7z x -y -o{SOURCE_DIR} {SOURCE_DIR}/train.7z > /dev/null
print('200k training file found:', os.path.exists(f'{SOURCE_DIR}/train.txt'))

Mounted at /content/drive
Cloning into 'pubmed-rct'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 19 (delta 5), reused 14 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 177.08 MiB | 16.44 MiB/s, done.
Resolving deltas: 100% (5/5), done.
Updating files: 100% (13/13), done.
200k training file found: True


In [4]:
import pandas as pd
import numpy as np
import unicodedata

def read_split(path):
    """Read one split file. Returns one row per sentence (same function as in 02_data_preparation)."""
    rows, sentences, abstract_id = [], [], None

    def save_abstract():
        n = len(sentences)
        for i, (label, text) in enumerate(sentences):
            rows.append({'abstract_id': abstract_id, 'sentence_idx': i, 'n_sentences': n,
                         'label': label, 'text': text})

    for line in open(path, encoding='utf-8'):
        line = line.rstrip('\n')
        if line.startswith('###'):
            abstract_id, sentences = line[3:], []
        elif line.strip() == '':
            save_abstract()
            sentences = []
        else:
            label, text = line.split('\t', 1)
            sentences.append((label, text))
    save_abstract()
    return pd.DataFrame(rows)

def normalise(text):
    text = unicodedata.normalize('NFC', text)
    return ' '.join(text.split())

train_200k = read_split(f'{SOURCE_DIR}/train.txt')
train_200k['text'] = train_200k.text.map(normalise)
print(f'200k train: {train_200k.abstract_id.nunique():,} abstracts | {len(train_200k):,} sentences')

200k train: 190,654 abstracts | 2,211,861 sentences


In [5]:
def load_ids(split):
    df = pd.read_csv(f'{PREPARED_DIR}/{EVAL_VERSION}_{split}.csv', dtype={'abstract_id': str},
                     keep_default_na=False, usecols=['abstract_id', 'text'])
    return set(df.abstract_id), set(df.text)

dev_ids, dev_texts = load_ids('dev')
test_ids, test_texts = load_ids('test')
eval_ids = dev_ids | test_ids

overlap = train_200k.abstract_id.isin(eval_ids)
print(f'Abstracts of our dev/test found inside the 200k training set: '
      f'{train_200k[overlap].abstract_id.nunique():,} (removed)')
train_filtered = train_200k[~overlap].reset_index(drop=True)

assert not set(train_filtered.abstract_id) & dev_ids, 'Leakage: dev abstract still in training data'
assert not set(train_filtered.abstract_id) & test_ids, 'Leakage: test abstract still in training data'
assert not set(train_filtered.label) - set(CLASSES), 'Unknown labels'
print(f'✅ Filtered training set: {train_filtered.abstract_id.nunique():,} abstracts | '
      f'{len(train_filtered):,} sentences | no dev/test abstract left')

train_texts = set(train_filtered.text)
for name, texts in [('dev', dev_texts), ('test', test_texts)]:
    pct = 100 * len(texts & train_texts) / len(texts)
    print(f'{name}: {pct:.1f}% of distinct sentences also appear word for word in the filtered training set')

Abstracts of our dev/test found inside the 200k training set: 4,862 (removed)
✅ Filtered training set: 185,792 abstracts | 2,153,197 sentences | no dev/test abstract left
dev: 1.6% of distinct sentences also appear word for word in the filtered training set
test: 1.5% of distinct sentences also appear word for word in the filtered training set


In [6]:
def add_columns(df):
    """Same columns as in 02_data_preparation."""
    df = df.copy()
    df['y'] = df.label.map(CLASSES.index)
    df['position'] = df.sentence_idx / (df.n_sentences - 1).clip(lower=1)
    df['position_bin'] = np.minimum((df.position * N_POSITION_BINS).astype(int), N_POSITION_BINS - 1)
    df['n_words'] = df.text.str.split().str.len()
    by_abstract = df.groupby('abstract_id', sort=False).text
    df['previous'] = by_abstract.shift(1).fillna('none')
    df['next'] = by_abstract.shift(-1).fillna('none')
    df['context'] = ('sentence ' + (df.sentence_idx + 1).astype(str) + ' of ' + df.n_sentences.astype(str)
                     + ' . previous: ' + df['previous'] + ' next: ' + df['next'])
    return df

train_filtered = add_columns(train_filtered)
train_filtered.to_csv(OUTPUT_PATH, index=False)
print('Saved:', OUTPUT_PATH)
print(train_filtered.label.value_counts(normalize=True).round(3).to_string())

Saved: /content/drive/MyDrive/exsu_project/prepared_data/PubMed_200k_RCT_train_filtered.csv
label
RESULTS        0.347
METHODS        0.327
CONCLUSIONS    0.154
BACKGROUND     0.088
OBJECTIVE      0.085
